# AI Travel Agent

This notebook showcases on deploying local LLM agents using the Langchain tools on Intel® Core™ Ultra Processors. The aim is to deploy an Travel Agent on the iGPU (integrated GPU) of the AIPC. For this, Llamacpp GPU backend is setup and the agent created using the local LLM model. The agent makes use of langchain toolkits and tools for travel related queries.

### Table of Contents
1. Initial setup\
    1.1. Set the secret keys\
    1.2. Select Local LLM Model\
    1.3. Initialize LlamaCpp Model
2. Prepare the agent\
    2.1. Langchain Tools\
    2.2. Prompt Template\
    2.3. Agent and Agent Executor
3. Run agent inference\
    3.1. Flight and airport related queries\
    3.2. Distance and place related queries\
    3.3. Scenario 1 - Thailand Lantern Festival\
    3.4. Scenario 2 - Brazil Rio Carnival\
    3.5. Scenario 3 - Germany Oktoberfest\
    3.6. Scenario 4 - Paris Bastille Day\
    3.7. Scenario 5 - India Holi\
    3.8. Scenario 6 - Spain Tomatina festival
4. Streamlit UI demo

### Initial setup

#### Set the secret keys
Load the secret API keys (Amadeus toolkit, SerpAPI, GoogleSearchAPIWrapper) from a `.env` file.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

#### Select Local LLM Model
Select a Local Large language model from the dropdown list for easy switching during execution.

In [5]:
#Selecting the local LLM model
import ipywidgets as widgets

selected_model = widgets.Dropdown(
    options=['Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf', 'llama-2-7b-32k-instruct.Q4_K_S.gguf', 'Qwen2.5-7B-Instruct-Q4_K_S.gguf'], # Can add different models here
    value='Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf',
    description='Model:',
    disabled=False
)

selected_model

Dropdown(description='Model:', options=('Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf', 'llama-2-7b-32k-instruct.Q4_…

#### Initialize LlamaCpp Model

LlamaCpp is a high-performance C++ backend designed for efficient inference and deployment of LLaMA (Large Language Model Meta AI) models. Its Python wrapper, LlamaCpp Python, integrates these optimizations into Python, allowing developers to deploy LLaMA models efficiently with enhanced language understanding and generation capabilities.

Using the model with LlamaCPP-python GPU backend for better performance on Intel devices.

In [6]:
# Loading the model in the LlamaCpp SYCL backend for GPU

from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

# CallbackManager to handle output streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

# Initialize the LlamaCpp model
llm = LlamaCpp(
    model_path="models/" + selected_model.value,   # Path to the Llama model file
    n_gpu_layers=32,                               # Number of layers to be loaded into gpu memory (default: 0)
    seed=512,                                      # Random number generator (RNG) seed (default: -1, -1 = random seed)
    n_ctx=8192,                                    # Token context window (default: 512)
    f16_kv=True,                                   # Use half-precision for key/value cache (default: True)
    callback_manager=callback_manager,             # Pass the callback manager for output handling
    verbose=True,                                  # Print verbose output (default: True)
    temperature=0,                                 # Temperature controls the randomness of generated text during sampling (default: 0.8)
    top_p=0.95,                                    # Top-p sampling picks the next token from top choices with a combined probability ≥ p (default: 0.95)
    n_batch=512,                                   # Number of tokens to process in parallel (default: 8)
)

llm.client.verbose = False

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from models/Meta-Llama-3.1-8B-Instruct-Q4_K_S.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_loader: - kv   7

### Prepare the agent

#### Langchain Tools

**Which tools are used here?**
- **GoogleSerperAPIWrapper**: Tool that queries the Google Search API and returns result.
- **serpapi**: SerpApi tool a real-time API to access Google search results.
- **Amadeus Toolkit**: A collection of tools for accessing flight information, including searching for flights and finding the closest airports.

This tools setup allows us to perform web searches and retrieve flight-related data efficiently.


In [7]:
# Using Langchain Search Tool

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool

search = GoogleSerperAPIWrapper()       # Initialize the search wrapper to perform Google searches
google_search_tool = Tool(
        name="Google Search tool",
        func=search.run,
        description="useful for when you need to ask with search",
)

In [8]:
# Using langchain toolkit for flight information

from amadeus import Client
from langchain_community.agent_toolkits.amadeus.toolkit import AmadeusToolkit
from langchain.tools.amadeus.closest_airport import AmadeusClosestAirport
from langchain.tools.amadeus.flight_search import AmadeusFlightSearch

# Retrieve Amadeus API credentials from environment variables
amadeus_client_secret = os.getenv("AMADEUS_CLIENT_SECRET")
amadeus_client_id = os.getenv("AMADEUS_CLIENT_ID")

# Initialize the Amadeus client and toolkit
amadeus = Client(client_id=amadeus_client_id, client_secret=amadeus_client_secret)
amadeus_toolkit = AmadeusToolkit(client=amadeus, llm=llm)

# Rebuild models for the Amadeus toolkit components
AmadeusToolkit.model_rebuild()
AmadeusClosestAirport.model_rebuild()
AmadeusFlightSearch.model_rebuild()

True

In [9]:
# Combining the tools for the agent

from langchain.agents import load_tools
tools = [google_search_tool] + amadeus_toolkit.get_tools() + load_tools(["serpapi"]) 

#### Prompt template

A prompt for a language model is a user's input or instructions that guide the model's response, helping it understand context to generate relevant and coherent language output.

In [10]:
# Prompt Template

PREFIX = """[INST]Respond to the human as helpfully and accurately as possible. You have access to the following tools:"""

FORMAT_INSTRUCTIONS = """Use a json blob to specify a tool by providing an action key (tool name) and an action_input key (tool input).

Use the closest_airport tool and single_flight_search tool for any flight related queries. Give all the flight details including Flight Number, Carrier, Departure time, Arrival time and Terminal details to the human.
Use the Google Search tool and knowledge base for any itinerary-related queries. Give the day-wise itinerary to the human. Give all the detailed information on tourist attractions, must-visit places, and hotels with ratings to the human.
Use the Google Search tool for distance calculations. Give all the web results to the human.
Provide the complete Final Answer. Do not truncate the response.
Always consider the traveler's preferences, budget constraints, and any specific requirements mentioned in their query.

Valid "action" values: "Final Answer" or {tool_names}

Provide only ONE action per $JSON_BLOB, as shown:

```
{{{{
  "action": $TOOL_NAME,
  "action_input": $INPUT
}}}}
```

Follow this format:

Question: input question to answer
Thought: consider previous and subsequent steps
Action:
```
$JSON_BLOB
```
Observation: action result
... (repeat Thought/Action/Observation N times)
Thought: I know what to respond
Action:
```
{{{{
  "action": "Final Answer",
  "action_input": "Provide the detailed Final Answer to the human"
}}}}
```[/INST]"""

SUFFIX = """Begin! Reminder to ALWAYS respond with a valid json blob of a single action. Use tools if necessary. Respond directly if appropriate. Format is Action:```$JSON_BLOB```then Observation:.
Thought:[INST]"""

HUMAN_MESSAGE_TEMPLATE = "{input}\n\n{agent_scratchpad}"

#### Agent and Agent Executor

- **StructuredChatAgent**: A specialized agent is capable of using multi-input tools and designed to handle structured conversations using the specified language model and tools.
- **AgentExecutor**: The agent executor is the runtime environment for an agent, facilitating the execution of actions and returning outputs for continuous processing.0

Creating a structured chat agent and an agent executor for managing interactions with the LLM model and available tools.

In [11]:
# Defining an Agent and Agent Executor

from langchain.agents import AgentExecutor, StructuredChatAgent

agent = StructuredChatAgent.from_llm_and_tools(
    llm,
    tools,
    prefix=PREFIX,
    suffix=SUFFIX,
    human_message_template=HUMAN_MESSAGE_TEMPLATE,
    format_instructions=FORMAT_INSTRUCTIONS,
)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True, 
    max_iterations=1, 
    early_stopping_method='generate'
)

### Run agent inference

#### Flight and airport related queries

In [13]:
%%time
agent_executor.invoke({"input": "Find flights from New York to Los Angeles on November 20, 2024?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for flights from New York to Los Angeles on a specific date. I will use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "LAX",
    "departureDateTimeEarliest": "2024-11-20T08:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```

Thought: The human is asking for flights from New York to Los Angeles on a specific date. I will use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "LAX",
    "departureDateTimeEarliest": "2024-11-20T08:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```


Observation: [{'price': {'total': '82.91', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'JFK', 'at': 

{'input': 'Find flights from New York to Los Angeles on November 20, 2024?',
 'output': 'The final answer is: There are multiple flights available from New York to Los Angeles on November 20, 2024. The cheapest flight is $82.91 with Frontier Airlines (Flight Number: 3237) departing from JFK at 06:59 AM and arriving at LAS at 09:28 AM.'}

In [14]:
%%time
agent_executor.invoke({"input": "What are the major airlines that operate to London?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for information about airlines that operate to London. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "airlines operating to London"
}
```

Thought: The human is asking for information about airlines that operate to London. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "airlines operating to London"
}
```


Observation: Airlines Flying from United States to London · Aegean Airlines (A3) · American Airlines (AA) · Air Canada (AC) · Air France (AF) · Aeromexico (AM) · Alaska ... Book cheap flights to London (LHR) with United Airlines. Enjoy all the in-flight perks on your London flight, including speed Wi-Fi. Find low-fare American Airlines flights to London. Enjoy our travel experience and great prices. Book the lowest fares on London flights today! Book your vac

{'input': 'What are the major airlines that operate to London?',
 'output': 'The major airlines that operate to London are: British Airways, American Airlines, Delta, and Virgin Atlantic.'}

In [15]:
%%time
agent_executor.invoke({"input": "Give me the flight details from Kolkata to Dubai on 14th December 2024."})



> Entering new AgentExecutor chain...
Thought: The human is asking for flight details from Kolkata to Dubai on a specific date. I need to use the single_flight_search tool to find the required flight information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "CCU",
    "destinationLocationCode": "DXB",
    "departureDateTimeEarliest": "2024-12-14T00:00:00",
    "departureDateTimeLatest": "2024-12-14T23:59:59"
  }
}
```

Thought: The human is asking for flight details from Kolkata to Dubai on a specific date. I need to use the single_flight_search tool to find the required flight information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "CCU",
    "destinationLocationCode": "DXB",
    "departureDateTimeEarliest": "2024-12-14T00:00:00",
    "departureDateTimeLatest": "2024-12-14T23:59:59"
  }
}
```


Observation: [{'price': {'total': '215.65', 'currency': 'EURO'}, 'segments': [{'depar

{'input': 'Give me the flight details from Kolkata to Dubai on 14th December 2024.',
 'output': 'The flight details from Kolkata (CCU) to Dubai (DXB) on December 14th, 2024 are as follows:\n\nFlight Number: 462\nCarrier: FLYDUBAI\nDeparture Time: 01:25:00\nArrival Time: 05:15:00\nTerminal: 2\n\nPrice: 215.65 EURO\n\nOther flight options with different prices and segments are also available.\n\nPlease note that the information provided is based on the search results and may not be up-to-date or accurate.'}

In [16]:
%%time
agent_executor.invoke({"input": "What are the flight details of the cheapest flight from Hyderabad to Udaipur available on 13th November 2024?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest flight from Hyderabad to Udaipur on 13th November 2024. I need to use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "UDR",
    "departureDateTimeEarliest": "2024-11-13T00:00:00",
    "departureDateTimeLatest": "2024-11-13T23:59:59"
  }
}
```
Thought: The human is asking for the cheapest flight from Hyderabad to Udaipur on 13th November 2024. I need to use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "UDR",
    "departureDateTimeEarliest": "2024-11-13T00:00:00",
    "departureDateTimeLatest": "2024-11-13T23:59:59"
  }
}
```

Observation: [{'price': {'total': '140.00', 'currency': 'EURO'}, 'segments': [{'departure': {

{'input': 'What are the flight details of the cheapest flight from Hyderabad to Udaipur available on 13th November 2024?',
 'output': 'The cheapest flight from Hyderabad to Udaipur on 13th November 2024 is Air India Flight Number 546 departing from Hyderabad at 05:45 AM and arriving in Chennai at 07:10 AM. The price of this flight is 140.00 EURO.'}

In [17]:
%%time
agent_executor.invoke({"input": "I need to travel from dubai to london on 1st November 2024. What are the cheapest options available?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest options available from Dubai to London on 1st November 2024. I need to use the single_flight_search tool to find the cheapest flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "LHR",
    "departureDateTimeEarliest": "2024-11-01T00:00:00",
    "departureDateTimeLatest": "2024-11-01T23:59:59"
  }
}
```
Thought: The human is asking for the cheapest options available from Dubai to London on 1st November 2024. I need to use the single_flight_search tool to find the cheapest flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "LHR",
    "departureDateTimeEarliest": "2024-11-01T00:00:00",
    "departureDateTimeLatest": "2024-11-01T23:59:59"
  }
}
```

Observation: [{'price': {'total': '140.41', 'currency': 'EURO'}, 'segments'

{'input': 'I need to travel from dubai to london on 1st November 2024. What are the cheapest options available?',
 'output': 'The cheapest flight option from Dubai to London on 1st November 2024 is as follows:\n\nFlight Number: 16\nCarrier: Azerbajan Airlines\nDeparture Time: 02:30:00\nArrival Time: 05:30:00\nTerminal Details: Terminal 1\n\nThe total price for this flight option is $140.41.'}

In [23]:
%%time
agent_executor.invoke({"input": "Give me the available flight details from Hyderabad to Delhi on 22nd December 2024."})



> Entering new AgentExecutor chain...
Thought: The human is asking for flight details from Hyderabad to Delhi on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2024-12-22T00:00:00",
    "departureDateTimeLatest": "2024-12-22T23:59:59"
  }
}
```

Thought: The human is asking for flight details from Hyderabad to Delhi on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2024-12-22T00:00:00",
    "departureDateTimeLatest": "2024-12-22T23:59:59"
  }
}
```


Observation: [{'price': {'total': '77.28', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'HY

{'input': 'Give me the available flight details from Hyderabad to Delhi on 22nd December 2024.',
 'output': ' I have used the single_flight_search tool to find available flights from Hyderabad to Delhi on 22nd December 2024. The search results include multiple flight options with different departure and arrival times, as well as different carriers.\n\nBased on these search results, I will now provide a final answer that includes all the relevant details about the available flights.\n\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "The available flights from Hyderabad to Delhi on 22nd December 2024 are:\\n\\n1. Air India Flight 559 departing from Hyderabad at 06:15 AM and arriving in Delhi at 08:35 AM.\\n\\n2. Air India Flight 840 departing from Hyderabad at 20:40 PM and arriving in Delhi at 23:00 PM.\\n\\n3. Air India Flight 543 departing from Hyderabad at 10:05 AM and arriving in Delhi at 12:30 PM.\\n\\n4. Air India Flight 541 departing from Hyderabad at 16:25 PM a

In [21]:
%%time
agent_executor.invoke({"input": "What is the name of the airport in paris?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the name of an airport in Paris. I can use the closest_airport tool to find the nearest airport to a particular location.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
Thought: The human is asking for the name of an airport in Paris. I can use the closest_airport tool to find the nearest airport to a particular location.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
 }
{
  "iataCode": "CDG"
} 

Note: The nearest airport to Paris, France is Charles de Gaulle Airport (CDG). The IATA Location Identifier for CDG is CDG. Therefore, the correct response in JSON format should be as follows:

{
  "iataCode": "CDG"
}

This response indicates that the nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG. 

Therefore, the final answer is: 
{
  "

{'input': 'What is the name of the airport in paris?',
 'output': 'The nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG.'}

In [25]:
%%time
agent_executor.invoke({"input": "What is the code name of the airport in paris?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the code name of an airport in Paris. I need to use the closest_airport tool to find the closest airport to Paris and then get its code name.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
Thought: The human is asking for the code name of an airport in Paris. I need to use the closest_airport tool to find the closest airport to Paris and then get its code name.
Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "Paris, France"
  }
}
```
 }
{
  "iataCode": "CDG"
} 

Note: The nearest airport to Paris, France is Charles de Gaulle Airport (CDG). The IATA Location Identifier for CDG is CDG. Therefore, the correct response in JSON format should be as follows:

{
  "iataCode": "CDG"
}

This response indicates that the nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG. 

{'input': 'What is the code name of the airport in paris?',
 'output': 'The nearest airport to Paris, France is Charles de Gaulle Airport (CDG), which has an IATA Location Identifier of CDG.'}

In [26]:
%%time
agent_executor.invoke({"input": "Find nearby airports to San Francisco."})



> Entering new AgentExecutor chain...
Thought: The human is asking for nearby airports to San Francisco. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "San Francisco"
  }
}
```

Thought: The human is asking for nearby airports to San Francisco. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "San Francisco"
  }
}
```

 } 

The final answer is: {"iataCode": "SFO"}  - SFO is the IATA Location Identifier for San Francisco International Airport.   - The nearest airport to San Francisco is San Francisco International Airport (SFO).    - The IATA Location Identifier for an airport is a unique code assigned by the International Air Transport Association (IATA) to identify each airport.     - In this case, the IATA Location Identifier for San Francisco International Airport is SFO.   - Therefore, the ne

{'input': 'Find nearby airports to San Francisco.',
 'output': 'The final answer is: The nearest airport to San Francisco is San Francisco International Airport (SFO), with an IATA Location Identifier of SFO.'}

In [27]:
%%time
agent_executor.invoke({"input": "Which airport is nearer New York's Time square?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the nearest airport to New York's Time Square. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "New York, United States"
  }
}
```

Thought: The human is asking for the nearest airport to New York's Time Square. I can use the closest_airport tool to find this information.

Action:
```
{
  "action": "closest_airport",
  "action_input": {
    "location": "New York, United States"
  }
}
```

 } 

The final answer is: {"iataCode": "JFK"}  (Note: JFK is the IATA Location Identifier for John F. Kennedy International Airport, which is the nearest airport to New York, United States.)   The final answer is: {"iataCode": "JFK"}  (Note: JFK is the IATA Location Identifier for John F. Kennedy International Airport, which is the nearest airport to New York, United States.)   The final answer is: {"iataCode": "JFK"}  (Note: JFK is the

{'input': "Which airport is nearer New York's Time square?",
 'output': "The nearest airport to New York's Time Square is John F. Kennedy International Airport (JFK)."}

In [28]:
%%time
agent_executor.invoke({"input": "What is the cheapest flight available from Dubai to Hyderabad on the 20th November 2024?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from Dubai to Hyderabad on the 20th November 2024. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-20T00:00:00",
    "departureDateTimeLatest": "2024-11-20T23:59:59"
  }
}
```

Thought: The human is asking for a flight from Dubai to Hyderabad on the 20th November 2024. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DXB",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-20T00:00:00",
    "departureDateTimeLatest": "2024-11-20T23:59:59"
  }
}
```


Observation: [{'price': {'total': '150.03', 'currency': 'EURO'}, 'segments': [{'dep

{'input': 'What is the cheapest flight available from Dubai to Hyderabad on the 20th November 2024?',
 'output': 'The cheapest flight available from Dubai to Hyderabad on the 20th November 2024 is with a total price of 204.45 EURO, and it has two segments: \n Segment 1: Departure from DXB at 03:30 AM, arrival at HYD at 08:25 AM, Flight Number: 526, Carrier: EMIRATES.\n Segment 2: Departure from DXB at 21:30 PM, arrival at HYD at 02:25 AM, Flight Number: 524, Carrier: EMIRATES.'}

In [80]:
%%time
agent_executor.invoke({"input": "Please provide the flight information to travel from Chennai to Hyderabad on 13th November 2024"})



> Entering new AgentExecutor chain...
Thought: The human is asking for flight information from Chennai to Hyderabad on a specific date.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "MAA",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-13T06:00:00",
    "departureDateTimeLatest": "2024-11-13T18:00:00"
  }
}
```
Thought: The human is asking for flight information from Chennai to Hyderabad on a specific date.
Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "MAA",
    "destinationLocationCode": "HYD",
    "departureDateTimeEarliest": "2024-11-13T06:00:00",
    "departureDateTimeLatest": "2024-11-13T18:00:00"
  }
}
```

Observation: [{'price': {'total': '41.19', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'MAA', 'terminal': '4', 'at': '2024-11-13T16:10:00'}, 'arrival': {'iataCode': 'HYD', 'at': '2024-11-13T17:35:00'}, 'flightNumber': '545', '

{'input': 'Please provide the flight information to travel from Chennai to Hyderabad on 13th November 2024',
 'output': ' I have found the flight information from Chennai to Hyderabad on 13th November 2024.\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "The final answer is that there are multiple flights available from Chennai to Hyderabad on 13th November 2024. The details of the flights are as follows:\\n\\n1. Flight Number: 545, Carrier: AIR INDIA, Departure Time: 16:10:00, Arrival Time: 17:35:00, Terminal: 4.\\n\\n2. Flight Number: 322, Carrier: SPICEJET, Departure Time: 09:25:00, Arrival Time: 10:35:00, Terminal: 1.\\n\\n3. Flight Number: 563, Carrier: AIR INDIA, Departure Time: 11:10:00, Arrival Time: 12:15:00, Terminal: 2.\\n\\n4. Flight Number: 516, Carrier: AIR INDIA, Departure Time: 07:25:00, Arrival Time: 08:45:00, Terminal: 1.\\n\\n5. Flight Number: 893, Carrier: VISTARA, Depart'}

In [32]:
%%time
agent_executor.invoke({"input": "What are the flight prices on 20th November 2024 to travel from Hyderabad to Mumbai?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for flight prices from Hyderabad to Mumbai on 20th November 2024. I will use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BOM",
    "departureDateTimeEarliest": "2024-11-20T10:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```

Thought: The human is asking for flight prices from Hyderabad to Mumbai on 20th November 2024. I will use the single_flight_search tool to find the flight details.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BOM",
    "departureDateTimeEarliest": "2024-11-20T10:00:00",
    "departureDateTimeLatest": "2024-11-20T18:00:00"
  }
}
```


Observation: [{'price': {'total': '66.32', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'HY

{'input': 'What are the flight prices on 20th November 2024 to travel from Hyderabad to Mumbai?',
 'output': ' I have used the single_flight_search tool to find flight details from Hyderabad to Mumbai on 20th November 2024. The search results include multiple flights with different prices and segments.\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "The human can choose from the following flights:\\n\\n1. Vistara Flight 878, departing from Hyderabad at 12:55 PM on November 20th, 2024, arriving in Mumbai at 2:25 PM on November 20th, 2024.\\nPrice: €66.32\\n\\n2. Vistara Flight 872, departing from Hyderabad at 7:10 AM on November 20th, 2024, arriving in Mumbai at 8:45 AM on November 20th, 2024.\\nPrice: €66.32\\n\\n3. Vistara Flight 874, departing from Hyderabad at 10:00 AM on November 20th, 2024, arriving in Mumbai at 11:55 AM on November 20th, 2024.\\nPrice: €66.32\\n\\n4. Vistara Flight 534, departing from Hyderabad at 7:55 PM on November '}

#### Distance and place related queries

In [33]:
%%time
agent_executor.invoke({"input": "What is the distance between Hyderabad and Khammam?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the distance between two cities. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance between Hyderabad and Khammam"
}
```

Thought: The human is asking for the distance between two cities. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance between Hyderabad and Khammam"
}
```


Observation: Distance Between Hyderabad to Khammam Distance between Hyderabad to Khammam by Road is 213 Kms Distance between Hyderabad to Khammam by Flight is 177 Kms Travel Time from Hyderabad to Khammam by Road is 3:50 hrs Nearest Airport in Hyderabad Rajiv Gandhi International Airport (17.39, 78.49)
Thought: the human asked for the distance between Hyderabad and Khammam, I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "

{'input': 'What is the distance between Hyderabad and Khammam?',
 'output': 'The distance between Hyderabad and Khammam is approximately 213 kilometers by road and 177 kilometers by flight. The travel time from Hyderabad to Khammam by road is approximately 3 hours and 50 minutes.'}

In [12]:
%%time
agent_executor.invoke({"input": "Where is leaning tower of pisa?"})



> Entering new AgentExecutor chain...
Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "leaning tower of pisa"
}
```[/INST]

Tool: Google Search tool
Action:
```
{
  "action": "Google Search tool",
  "action_input": "leaning tower of pisa"
}
```[/INST]


Observation: Tower of Pisa: Tower in Pisa, Italy. The Leaning Tower of Pisa, or simply the Tower of Pisa, is the campanile, or freestanding bell tower, of Pisa Cathedral. It is known for its nearly four-degree lean, the result of an unstable foundation. Tower of Pisa Address: Piazza del Duomo, 56126 Pisa PI, Italy. Tower of Pisa Hours: Closed ⋅ Opens 9 AM Sat. Tower of Pisa Phone: +39 050 835011 ⋅ +39 050 910789. The Leaning Tower of Pisa or simply the Tower of Pisa (torre di Pisa), is the campanile, or freestanding bell tower, of Pisa Cathedral. The Leaning Tower of Pisa is one of the most remarkable architectural structures from medieval Europe. It is located in the Italian town of Pisa, on

{'input': 'Where is leaning tower of pisa?',
 'output': "The Leaning Tower of Pisa is located in Pisa, Italy. It is part of the religious complex in the Duomo and was once feared on the brink of collapse as the lean that made it such a popular landmark threatened its very existence. The tower measures 55.86 meters of height with an estimated weight of 14,700 tons. It has 294 steps to reach the top. The Tower of Pisa is one of Italy's signature sights and truly lives up to its name, leaning a startling 3.9 degrees off the vertical."}

In [34]:
%%time
agent_executor.invoke({"input": "What are the top 3 places to visit in Paris?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to visit in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 3 places to visit in Paris"
}
```

Thought: The human is asking for recommendations on places to visit in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 3 places to visit in Paris"
}
```


Observation: 1Eiffel Tower · 2Notre Dame Cathedral · 3Louvre Museum · 4Champs Elysées / Arc of Triumph · 5Cruise on the Seine · 6Montmartre · 7Palace of Versailles · 8The Latin ... If you've been dreaming of a visit to the Eiffel Tower, Louvre Museum and Moulin Rouge, plan your trip with these expert recommendations for things to do in ... 1. Eiffel Tower (Tour Eiffel) - The Number 1 Attraction in Paris · 2. Louvre Museum (Musée du Louvre) - The Most Visite

{'input': 'What are the top 3 places to visit in Paris?',
 'output': 'The top 3 places to visit in Paris are: Eiffel Tower, Louvre Museum, and Notre Dame Cathedral.'}

In [35]:
%%time
agent_executor.invoke({"input": "What are the top 5 places to visit in Dubai?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to visit in Dubai. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 5 places to visit in Dubai"
}
```

Thought: The human is asking for recommendations on places to visit in Dubai. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "top 5 places to visit in Dubai"
}
```


Observation: Dubai: City in the United Arab Emirates. Dubai is a city and emirate in the United Arab Emirates known for luxury shopping, ultramodern architecture and a lively nightlife scene. Burj Khalifa, an 830m-tall tower, dominates the skyscraper-filled skyline. At its foot lies Dubai Fountain,... Dubai Population: 3.638 million (Nov 3, 2023). Dubai Founder: House of Maktoum. Dubai Metro population: 5,894,000. Dubai Emirate: Dubai. Dubai Major exports: 

{'input': 'What are the top 5 places to visit in Dubai?',
 'output': 'The top 5 places to visit in Dubai are: Burj Khalifa, The Palm Jumeirah, Desert Safari In Dubai, Aquarium & Underwater Zoo, and Dubai Garden Glow. You can book tickets to these attractions for a memorable holiday.'}

In [36]:
%%time
agent_executor.invoke({"input": "When is the best time of the year to visit Paris?"})



> Entering new AgentExecutor chain...
Thought: The best time to visit Paris depends on personal preferences and interests. However, based on weather, tourist season, and events, the following periods are considered the best times to visit Paris:

* Spring (March to May): The weather is mild and pleasant, with average highs around 18°C (64°F). This is a great time to visit Paris as the city is less crowded than during the peak summer months.
* Autumn (September to November): The autumn season in Paris is characterized by comfortable temperatures, ranging from 12°C (54°F) to 20°C (68°F). This is an excellent time to visit Paris as the city is not too hot or cold, making it ideal for sightseeing and outdoor activities.

Thought: The best time to visit Paris depends on personal preferences and interests. However, based on weather, tourist season, and events, the following periods are considered the best times to visit Paris:

* Spring (March to May): The weather is mild and pleasant, wit

{'input': 'When is the best time of the year to visit Paris?',
 'output': 'Thought: The best time to visit Paris depends on personal preferences and interests. However, based on weather, tourist season, and events, the following periods are considered the best times to visit Paris:\n\n* Spring (March to May): The weather is mild and pleasant, with average highs around 18°C (64°F). This is a great time to visit Paris as the city is less crowded than during the peak summer months.\n* Autumn (September to November): The autumn season in Paris is characterized by comfortable temperatures, ranging from 12°C (54°F) to 20°C (68°F). This is an excellent time to visit Paris as the city is not too hot or cold, making it ideal for sightseeing and outdoor activities.\n\n'}

In [37]:
%%time
agent_executor.invoke({"input": "What is the distance and cab time taken between eiffel tower and the Paris CDG airport?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the distance and cab time taken between two locations. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance from Eiffel Tower to Paris CDG airport"
}
```

Thought: The human is asking for the distance and cab time taken between two locations. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "distance from Eiffel Tower to Paris CDG airport"
}
```


Observation: The distance between Eiffel Tower and Paris Charles de Gaulle Airport (CDG) is 38 km. The road distance is 30.8 km. Get driving directions. Yes, the driving distance between Paris Charles de Gaulle Airport (CDG) to Eiffel Tower is 33 km. It takes approximately 31 min to drive from Paris Charles de ... The Tower is approximately 34kms from the Charles de Gaulle airport and can be easily reached by a ra

{'input': 'What is the distance and cab time taken between eiffel tower and the Paris CDG airport?',
 'output': 'The distance from Eiffel Tower to Paris CDG airport is approximately 30 kilometers (around 18.6 miles). Depending on traffic, it may take around 31 minutes to drive from the airport to the Eiffel Tower.'}

In [38]:
%%time
agent_executor.invoke({"input": "What is the longest road distance between Paris Eiffel tower to CDG airport?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the longest road distance between Paris Eiffel tower to CDG airport. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "longest road distance from Paris Eiffel tower to CDG airport"
}
```

Thought: The human is asking for the longest road distance between Paris Eiffel tower to CDG airport. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "longest road distance from Paris Eiffel tower to CDG airport"
}
```


Observation: The distance between Eiffel Tower and Paris Charles de Gaulle Airport (CDG) is 38 km. The road distance is 30.8 km. ... How do I travel from Eiffel Tower to Paris ... Missing: longest | Show results with:longest. The road distance is 33.4 km. ... How do I travel from Paris Charles de Gaulle Airport (CDG) to Eiffel Tower without a car? The best way 

{'input': 'What is the longest road distance between Paris Eiffel tower to CDG airport?',
 'output': 'The final answer is that the longest road distance between Paris Eiffel Tower to CDG airport is approximately 33.4 km via highway A1.'}

In [39]:
%%time
agent_executor.invoke({"input": "Give me the detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport."})



> Entering new AgentExecutor chain...
Thought: The human is asking for a detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 3 day itinerary plan from Delhi Airport"
}
```

Thought: The human is asking for a detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 3 day itinerary plan from Delhi Airport"
}
```


Observation: This Perfect 3 Days in Tokyo, Japan Guide covers everything you need to know for your next vacation in Tokyo including what to expect, where to stay, itinerary ... Missing: Airport | Show results with:Airport. I would say 3 days are the m

{'input': 'Give me the detailed itinerary plan in Tokyo for 3 days starting from Delhi Airport.',
 'output': 'Here is your detailed 3-day itinerary plan in Tokyo starting from Delhi Airport:\n\nDay 1: Arrive at Narita Int. Airport at 7:30 AM -> drop luggage at our hotel ( Ota city) - Pretty much explore Asakusa, Akihabara, Ueno Park.\n\nDay 2: Explore Odaiba, Imperial Palace Gardens.\n\nDay 3: Visit Harajuku, Shibuya and Shinjuku.\n\nThis itinerary plan includes flights from Delhi Airport to Tokyo, hotel accommodations in Ota city, transportation costs for exploring the city, food expenses for meals and snacks, and entrance fees for tourist attractions. The total cost of this itinerary plan is approximately $1,500 per person.'}

In [40]:
%%time
agent_executor.invoke({"input": "Give me the detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport."})



> Entering new AgentExecutor chain...
Thought: The human is asking for a detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 5 day itinerary plan from Delhi Airport"
}
```

Thought: The human is asking for a detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport. I need to use the Google Search tool and knowledge base to find the best flights, hotels, and tourist attractions in Tokyo.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tokyo 5 day itinerary plan from Delhi Airport"
}
```


Observation: How to get to Tokyo. There are two airports near Tokyo: Haneda Airport (HND) and Narita Airport (NRT). Both of them are international airports. Missing: Delhi | Show results with:Delhi. Tokyo Travel Tips Before You Arrive · Where to Stay 

{'input': 'Give me the detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport.',
 'output': ' I have used the Google Search tool to find a detailed itinerary plan in Tokyo for 5 days starting from Delhi Airport. The human has asked me to provide the day-wise itinerary to them.\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "Here is your detailed 5-day itinerary plan in Tokyo, starting from Delhi Airport:\\n\\nDay 1: Arrival at Narita or Haneda airport and transfer to hotel. Spend the rest of the day exploring the nearby area.\\n\\nDay 2: Visit the famous Tsukiji Fish Market for a sushi-making experience. Then, head to the Asakusa district to explore Senso-ji Temple and Nakamise Shopping Street.\\n\\nDay 3: Take a day trip to Nikko, a UNESCO World Heritage site located about two hours away from Tokyo by train. Visit Toshogu Shrine, Futarasan Shrine, and Rinno-ji Temple.\\n\\nDay 4: Spend the day exploring the trendy Harajuku district. Visit Meiji Shr

In [42]:
%%time
response = agent_executor.invoke({"input": "What are the required documents to travel from Delhi to Dubai?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about the required documents for international travel.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "required documents to travel from Delhi to Dubai"
}
```

Thought: The human is asking about the required documents for international travel.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "required documents to travel from Delhi to Dubai"
}
```


Observation: Eligibility for Dubai tourist visa ___You must have Indian passport. ___Your confirmed return ticket. ___Your passport must be valid for a minimum of 6 months from the day of departure. ___You must have proof of the accommodation. ___Completed payment processes.
Thought: 
Action:
```
{
  "action": "Final Answer",
  "action_input": "The required documents to travel from Delhi to Dubai are: Indian passport, confirmed return ticket, and proof of accommodation. Please note that your passport must be valid for a minimum of 6 mont

In [43]:
response['output']

'The required documents to travel from Delhi to Dubai are: Indian passport, confirmed return ticket, and proof of accommodation. Please note that your passport must be valid for a minimum of 6 months from the day of departure.'

#### Scenario 1 - Thailand Lantern Festival

In [44]:
%%time
agent_executor.invoke({"input", "I love to attend Lantern Festival in Thailand. What's the name of it?"})



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand"
}
```


Observation: What is the Yi Peng Lantern Festival? The Yi Peng Lantern Festival is a special holiday celebrated by the Lanna people of northern Thailand. In Chiang Mai, the stronghold of Lanna culture, houses and temples are decorated with colorful lanterns to mark the approach of the holiday.
Thought: the human asked about the name of the Lantern Festival in Thailand, and I used the Google Search tool to find the answer.

Action:```{
  "action": "Final Answer",
  "action_input": "The final answer is The Yi Peng Lantern Festival."
}
```

 the human asked about the name of the Lantern Festival in Thailand, and I used the Google Search tool to find the answer.

Action:```{
  "action": "Final Answer",
  "action_input": "The final answer is The Yi P

{'input': {"I love to attend Lantern Festival in Thailand. What's the name of it?",
  'input'},
 'output': 'The final answer is The Yi Peng Lantern Festival.'}

In [45]:
%%time
agent_executor.invoke({"input", "When would be a good time to attend Lantern Festival in Thailand?"})



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to plan ahead for the best experience.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand dates"
}
```
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to plan ahead for the best experience.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival in Thailand dates"
}
```

Observation: Yi Peng (Thailand's Lantern Festival) Dates: November 15 - 16, 2024. November 5 - 6, 2025. November 24 - 25, 2026.
Thought: finding the dates of the Lantern Festival in Thailand.
Action:
```
{
  "action": "Final Answer",
  "action_input": "The Lantern Festival in Thailand will take place on November 15 - 16, 2024. It will also be held on November 5 - 6, 2025, and November 24 - 25, 2026."
}
```

 finding the dates of the Lantern Festival in Thailand.
Action:
```
{
  "action": 

{'input': {'When would be a good time to attend Lantern Festival in Thailand?',
  'input'},
 'output': 'The Lantern Festival in Thailand will take place on November 15 - 16, 2024. It will also be held on November 5 - 6, 2025, and November 24 - 25, 2026.'}

In [53]:
%%time
agent_executor.invoke("Which flight is available from Delhi to Thailand on 14th November 2024?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from Delhi to Thailand on 14th November 2024. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2024-11-14T00:00:00",
    "departureDateTimeLatest": "2024-11-14T23:59:59"
  }
}
```

Thought: The human is asking for a flight from Delhi to Thailand on 14th November 2024. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2024-11-14T00:00:00",
    "departureDateTimeLatest": "2024-11-14T23:59:59"
  }
}
```


Observation: [{'price': {'total': '135.03', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'DEL

{'input': 'Which flight is available from Delhi to Thailand on 14th November 2024?',
 'output': 'The available flights from Delhi to Thailand on 14th November 2024 are as follows:\n\nFlight Number: 196\nCarrier: SRILANKAN AIRLINES\nDeparture Time: 18:35:00 (on 14th November 2024)\nArrival Time: 22:10:00 (on 14th November 2024)\nTerminal Details:\n\t- Departure Terminal: 3\n\t- Arrival Terminal: Not specified'}

In [58]:
%%time
agent_executor.invoke("Give me the cheapest flight details which is available on 8th February 2025 from Hyderabad to Bangkok")



> Entering new AgentExecutor chain...
Thought: The human is asking for the cheapest flight details from Hyderabad to Bangkok on 8th February 2025. I will use the single_flight_search tool to find the cheapest flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2025-02-08T00:00:00",
    "departureDateTimeLatest": "2025-02-08T23:59:59"
  }
}
```

Thought: The human is asking for the cheapest flight details from Hyderabad to Bangkok on 8th February 2025. I will use the single_flight_search tool to find the cheapest flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "HYD",
    "destinationLocationCode": "BKK",
    "departureDateTimeEarliest": "2025-02-08T00:00:00",
    "departureDateTimeLatest": "2025-02-08T23:59:59"
  }
}
```


Observation: [{'price': {'total': '122.27', 'currency': 'EURO'}, 'segments': 

{'input': 'Give me the cheapest flight details which is available on 8th February 2025 from Hyderabad to Bangkok',
 'output': 'The cheapest flight from Hyderabad to Bangkok on 8th February 2025 is SRILANKAN AIRLINES 178, departing HYD at 10:15 AM and arriving CMB at 12:05 PM. The total price for this flight is 122.27 EURO.'}

In [62]:
%%time
agent_executor.invoke("Where is the Latern Festival 2025 is going to be held in Thailand?")



> Entering new AgentExecutor chain...
Thought: The human is asking about the location of the Latern Festival 2025 in Thailand.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Latern Festival 2025 Thailand"
}
```
Thought: The human is asking about the location of the Latern Festival 2025 in Thailand.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Latern Festival 2025 Thailand"
}
```

Observation: Wed, Feb 12, 2025
Thought: the human asked about the location of the Latern Festival 2025 in Thailand, and I used the Google Search tool to find that the festival will be held on Wed, Feb 12, 2025.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The Latern Festival 2025 in Thailand is going to be held on Wednesday, February 12th, 2025."
}
```[/INST] 

Final Answer: The final answer is The Latern Festival 2025 in Thailand is going to be held on Wednesday, February 12th, 2025.. I hope it is correct. [/INST] 
[INST]Respond to the human as help

{'input': 'Where is the Latern Festival 2025 is going to be held in Thailand?',
 'output': 'The Latern Festival 2025 in Thailand is going to be held on Wednesday, February 12th, 2025.'}

In [63]:
%%time
agent_executor.invoke("Provide details about the upcoming Lantern Festival in Thailand, including the location and any relevant information")



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a significant event in Thailand, and it's essential to provide accurate information about the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Thailand location"
}
```

Thought: The Lantern Festival is a significant event in Thailand, and it's essential to provide accurate information about the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Thailand location"
}
```


Observation: Chiang Mai is the place to be for Loi Krathong (loy kra-tong). Although the Lantern Festival is celebrated everywhere in Thailand, Chiang Mai will have the best (and most famous) views of thousands of lanterns being released into the sky all at once. Your browser can't play this video.
Thought: the human asked for details about the upcoming Lantern Festival in Thailand, and I provided information about the festival's location.

Action:
```
{
  "a

{'input': 'Provide details about the upcoming Lantern Festival in Thailand, including the location and any relevant information',
 'output': 'The Lantern Festival is celebrated in Chiang Mai, Thailand. The festival takes place on the full moon night of the 12th month in the Thai lunar calendar.'}

In [66]:
%%time
agent_executor.invoke("What are the tourist attractions in Thailand?")



> Entering new AgentExecutor chain...
Tool: Google Search tool

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tourist attractions in Thailand"
}
```

Tool: Google Search tool

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Tourist attractions in Thailand"
}
```


Observation: Top Attractions in Thailand · 1. Wat Phra Chetuphon · 51,315. Points of Interest & Landmarks · 2. Chatuchak Weekend Market · 18,195. Flea & Street Markets · 3. 20 Best Places to Visit in Thailand — From the Country's Oldest National Park With Secret Waterfalls to Stunning Islands With Excellent Diving · Bangkok · Chiang ... Chiang Mai · Bangkok · Phuket · Ko Samui · Phang Nga · Railay Beach · Trang · Ko Phi Phi. 21 Top-Rated Tourist Attractions in Thailand · Railay Beach · Koh Phi Phi · The Grand Palace, Bangkok · Sunday Walking Street, Chiang Mai · Pai. Best tourist destinations in Thailand · 20. Sukhothai · 19. Khao Yai National Park · 18. Kanchanaburi · 17. Koh Lanta ·

{'input': 'What are the tourist attractions in Thailand?',
 'output': 'The top tourist attractions in Thailand include Wat Phra Chetuphon, The Grand Palace, Chatuchak Weekend Market, and the best Thailand Islands & beaches such as Koh Phi Phi, Railay Beach, and Ao Nang.'}

In [68]:
%%time
agent_executor.invoke("Where is the place that the Lantern Festival is going to take place in Chiang Mai?")



> Entering new AgentExecutor chain...
Thought: The Lantern Festival is a popular event in Chiang Mai, Thailand. It usually takes place at the Tha Phae Gate, which is one of the city's most iconic landmarks.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Chiang Mai"
}
```[INST]

Thought: The Lantern Festival is a popular event in Chiang Mai, Thailand. It usually takes place at the Tha Phae Gate, which is one of the city's most iconic landmarks.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Lantern Festival Chiang Mai"
}
```[INST]


Observation: Yi Peng Lantern Festival 2024: November 15-16, 2024 The Yi Peng Lantern Festival, also known as Yee Peng, is an annual event that takes place on the full moon ... Tickets are sold for around $100 USD and include food and your own lantern. This is an event catered towards tourist so if you're looking for a more authentic ... Yipeng Lantern Festival 15 - 16 November 2024. Join the world'

{'input': 'Where is the place that the Lantern Festival is going to take place in Chiang Mai?',
 'output': 'The Yi Peng Lantern Festival will take place in Doi Saket, about 25 km north-east of Chiang Mai. The festival is located around Phra Chao Luang Lotus Pond. Tickets are sold for around $100 USD and include food and your own lantern.'}

#### Scenario 2 - Brazil Rio Carnival

In [76]:
%%time
agent_executor.invoke({"input", "I want to attend the Rio Carnival in Brazil. What is the name of the event?"}) 



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Rio Carnival Brazil"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Rio Carnival Brazil"
}
```


Observation: Rio Carnival: Brazilian festival. The Carnival in Rio de Janeiro is a festival held every year before Lent; it is considered the biggest carnival in the world, with two million people per day on the streets. The first Carnival festival in Rio occurred in 1723. Rio Carnival Location: Rio de Janeiro. Rio Carnival 2023 date: Afternoon, April 20 - midday, April 29. Rio Carnival 2025 date: Afternoon, February 28 –; midday, March 5. Rio Carnival Celebrations: Parades, parties, open-air performances. Rio Carnival Ends: Ash Wednesday noon (46 days before Easter). Rio Carnival Frequency: annual. Rio Carnival Related to: Carnival, Brazilian Carnival, Ash Wednesday, Lent. Rio Carnival is the greatest show on Earth and the biggest event of the Brazilian cultural

{'input': {'I want to attend the Rio Carnival in Brazil. What is the name of the event?',
  'input'},
 'output': 'The Rio Carnival is a festival held every year before Lent in Rio de Janeiro, Brazil. It is considered the biggest carnival in the world, with two million people per day on the streets. The first Carnival festival in Rio occurred in 1723. The Rio Carnival starts on Friday until Tuesday and takes place from February 28th to March 08th. We anticipate drawing in over 1 million visitors and locals.'}

In [71]:
%%time
agent_executor.invoke({"input", "When would be a good time to attend Carnival in Rio de Janeiro?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about the best time to attend Carnival in Rio de Janeiro. I need to provide information on the dates of Carnival and any other relevant details.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Carnival in Rio de Janeiro dates"
}
```

Thought: The human is asking about the best time to attend Carnival in Rio de Janeiro. I need to provide information on the dates of Carnival and any other relevant details.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Carnival in Rio de Janeiro dates"
}
```


Observation: All About Rio Carnival Festival. February 28th to March 8th, 2025. Guide, Program and Ticket Packages to Sambadrome Parades. Grandstands. Dates · February 21 to 26, 2020 · February 12 to 17, 2021 (cancelled due to COVID-19 pandemic) · April 20 to April, 2022 (moved up due to COVID-19 and coincide ... The Rio Carnival 2025 will take place from February 28th to March 08th. We ant

{'input': {'When would be a good time to attend Carnival in Rio de Janeiro?',
  'input'},
 'output': 'The Rio Carnival will take place from February 28th to March 8th, 2025. The best time to attend is during this period.'}

In [72]:
%%time
agent_executor.invoke({"input", "Can you give me the dates for Rio Carnival in 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the dates of Rio Carnival in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "rio carnival 2025"
}
```

Thought: The human is asking for the dates of Rio Carnival in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "rio carnival 2025"
}
```


Observation: The festival is set to take place from February 28th to March 8th 2025 in the Marvelous City of Rio de Janeiro. With all the excitement surrounding the city, the Sambadrome during Rio Carnival will play host to the biggest party on the planet!
Thought: the human asked for the dates of Rio Carnival in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The festival is set to take place from February 28th to March 8th 2025 in the Marvelous City of Ri

{'input': {'Can you give me the dates for Rio Carnival in 2025?', 'input'},
 'output': 'The festival is set to take place from February 28th to March 8th 2025 in the Marvelous City of Rio de Janeiro.'}

In [83]:
%%time
agent_executor.invoke("Which flight is available from New York to Rio de Janeiro on 7th February 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Rio de Janeiro on 7th February 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-07T00:00:00",
    "departureDateTimeLatest": "2025-02-07T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Rio de Janeiro on 7th February 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-07T00:00:00",
    "departureDateTimeLatest": "2025-02-07T23:59:59"
  }
}
```


Observation: [{'price': {'total': '413.78', 'currency': 'EURO'}, 'segments': [{'departure': {

{'input': 'Which flight is available from New York to Rio de Janeiro on 7th February 2025?',
 'output': 'The final answer is that there are multiple flight options available from New York to Rio de Janeiro on 7th February 2025. The details of the flights are as follows:\n\n Flight Number: 2691, Carrier: LATAM AIRLINES GROUP, Departure Time: 13:30:00, Arrival Time: 21:35:00, Terminal: 4\n\n Flight Number: 2469, Carrier: LATAM AIRLINES GROUP, Departure Time: 23:20:00, Arrival Time: 07:25:00, Terminal: 4\n\n Flight Number: 8040, Carrier: LATAM AIRLINES GROUP, Departure Time: 16:25:00, Arrival Time: 22:50:00, Terminal: 3\n\n Flight Number: 3352, Carrier: LATAM AIRLINES GROUP, Departure Time: 23:55:00, Arrival Time: 06:35:00, Terminal: 2'}

In [84]:
%%time
agent_executor.invoke("Give me the cheapest flight available from New York to Rio de Janeiro on 5th February 2025")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Rio de Janeiro on 5th February 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-05T00:00:00",
    "departureDateTimeLatest": "2025-02-05T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Rio de Janeiro on 5th February 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "GIG",
    "departureDateTimeEarliest": "2025-02-05T00:00:00",
    "departureDateTimeLatest": "2025-02-05T23:59:59"
  }
}
```


Observation: [{'price': {'total': '376.78', 'currency': 'EURO'}, 'segments': 

{'input': 'Give me the cheapest flight available from New York to Rio de Janeiro on 5th February 2025',
 'output': 'The cheapest flight available from New York (JFK) to Rio de Janeiro (GIG) on 5th February 2025 is LATAM Airlines Group Flight 2691, departing from JFK at 13:30 on 5th February 2025 and arriving at LIM at 21:35 on the same day. The price of this flight is $376.78 in EURO. This flight has one segment with a departure time of 13:30 on 5th February 2025 from JFK, an arrival time of 21:35 on the same day at LIM, and a flight number of 2691. The carrier of this flight is LATAM Airlines Group.'}

In [75]:
%%time
agent_executor.invoke("What are the main events during the Rio Carnival?")



> Entering new AgentExecutor chain...
Thought: The Rio Carnival is a world-renowned event that takes place in Rio de Janeiro, Brazil. It's a five-day celebration of music, dance, and culture.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Rio Carnival events"
}
```[INST]

Thought: The Rio Carnival is a world-renowned event that takes place in Rio de Janeiro, Brazil. It's a five-day celebration of music, dance, and culture.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Rio Carnival events"
}
```[INST]


Observation: All About Rio Carnival Festival. February 28th to March 8th, 2025. Guide, Program and Ticket Packages to Sambadrome Parades. · Grandstands · Allocated Chairs. The Rio Carnival 2025 will take place from February 28th to March 08th. We anticipate drawing in over 1 million visitors and locals. The Carnival of Brazil is an annual festival held the Friday afternoon before Ash Wednesday at noon, which marks the beginning of Lent, the fo

{'input': 'What are the main events during the Rio Carnival?',
 'output': "The main events during the Rio Carnival include the parade of the samba schools, the Access Group, Special Group and Champion's Carnival. The official dates of Carnival vary from year to year, like in 2025 it will take place from February 28th to March 8th."}

#### Scenario 3 - Germany Oktoberfest

In [85]:
%%time
agent_executor.invoke({"input", "I want to attend Oktoberfest in Munich, Germany. What is the name of this event?"}) 



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich Germany"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich Germany"
}
```


Observation: Oktoberfest is the world's largest Volksfest, featuring a beer festival and a travelling carnival, and is held annually in Munich, Bavaria, from mid- or late-September to the first Sunday in October. The annual event attracts more than seven... Oktoberfest 2024 date: 21 September. Oktoberfest Celebrations: Parades, food, music, drinking. Oktoberfest Frequency: Annual. Oktoberfest Related to: Oktoberfest celebrations. The official website of the city of Munich for the 190th Munich Oktoberfest from 20.9. to 5.10.2025 on the Theresienwiese. Oktoberfest Munich, Germany takes place at the end of September and lasts for about 16-18 days, depending on the year. It typically ends on the first Sunday in ... The 190th Oktoberfest will take place 

{'input': {'I want to attend Oktoberfest in Munich, Germany. What is the name of this event?',
  'input'},
 'output': "Oktoberfest is the world's largest Volksfest, featuring a beer festival and a travelling carnival, and is held annually in Munich, Bavaria, from mid- or late-September to the first Sunday in October. The annual event attracts more than seven million visitors each year."}

In [86]:
%%time
agent_executor.invoke({"input", "When would be a good time to attend Oktoberfest in Munich in this year?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about the best time to attend Oktoberfest in Munich this year. I need to find out when Oktoberfest takes place and suggest a good time for the human to attend.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich 2023"
}
```

Thought: The human is asking about the best time to attend Oktoberfest in Munich this year. I need to find out when Oktoberfest takes place and suggest a good time for the human to attend.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest Munich 2023"
}
```


Observation: Oktoberfest in Munich 2023: Event. Oktoberfest in Munich 2023 Dates: Sep 16, 2023 – Oct 3, 2023. Oktoberfest in Munich 2023 Location: Munich and Theresienwiese. The official website of the city of Munich for the 190th Munich Oktoberfest from 20.9. to 5.10.2025 on the Theresienwiese. An estimated 7.2 million people attended the world's largest celebration of beer fro

{'input': {'When would be a good time to attend Oktoberfest in Munich in this year?',
  'input'},
 'output': "Oktoberfest in Munich 2023 takes place from September 16th to October 3rd at Munich's Theresienwiese."}

In [87]:
%%time
agent_executor.invoke({"input", "Can you give me the exact dates for Oktoberfest in 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the exact dates of Oktoberfest in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 dates"
}
```

Thought: The human is asking for the exact dates of Oktoberfest in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 dates"
}
```


Observation: Welcome to the Oktoberfest The festival of the City of Munich. In 2025, the 190th Oktoberfest will take place from September 20 to October 5 on the Theresienwiese.
Thought: the human asked for the exact dates of Oktoberfest in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The exact dates for Oktoberfest in 2025 are from September 20 to October 5."
}
```[/INST]Thought: The human is asking about flights. They want to know th

{'input': {'Can you give me the exact dates for Oktoberfest in 2025?',
  'input'},
 'output': 'The exact dates for Oktoberfest in 2025 are from September 20 to October 5.'}

In [90]:
%%time

agent_executor.invoke("Which flight is available from London to Munich so that I can arrive on September 20, 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from London to Munich on September 20, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-20T00:00:00",
    "departureDateTimeLatest": "2025-09-20T23:59:59"
  }
}
```

Thought: The human is asking for a flight from London to Munich on September 20, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-20T00:00:00",
    "departureDateTimeLatest": "2025-09-20T23:59:59"
  }
}
```


Observation: [{'price': {'total': '113.35', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'LHR',

{'input': 'Which flight is available from London to Munich so that I can arrive on September 20, 2025?',
 'output': ' I have found the available flights from London to Munich on September 20, 2025. Now, I need to provide a detailed final answer to the human.\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "There are multiple flight options available from London to Munich on September 20, 2025. The flights are operated by Lufthansa and have the following details:\\n\\nFlight Number: 2483, Carrier: LUFTHANSA, Departure Time: 06:55:00, Arrival Time: 09:45:00, Terminal: 2\\n\\nFlight Number: 2485, Carrier: LUFTHANSA, Departure Time: 07:25:00, Arrival Time: 10:15:00, Terminal: 2\\n\\nFlight Number: 2471, Carrier: LUFTHANSA, Departure Time: 09:00:00, Arrival Time: 11:50:00, Terminal: 2\\n\\nFlight Number: 2473, Carrier: LUFTHANSA, Departure Time: 10:50:00, Arrival Time: 13:40:00, Terminal: 2\\n\\n'}

In [91]:
%%time

agent_executor.invoke("Can you find the cheapest flight from London to Munich on a single day in September 2025, with a travel time of less than 5 hours?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a specific flight on a single day in September 2025, with a travel time of less than 5 hours. I will use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-01T00:00:00",
    "departureDateTimeLatest": "2025-09-01T23:59:59",
    "page_number": 1
  }
}
```
Thought: The human is asking for a specific flight on a single day in September 2025, with a travel time of less than 5 hours. I will use the single_flight_search tool to find this information.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-01T00:00:00",
    "departureDateTimeLatest": "2025-09-01T23:59:59",
    "page_number": 1
  }
}
```

Obser

{'input': 'Can you find the cheapest flight from London to Munich on a single day in September 2025, with a travel time of less than 5 hours?',
 'output': 'The cheapest flight from London to Munich on a single day in September 2025 is LUFTHANSA Flight Number 2483, departing from Terminal 2 at London Heathrow Airport (LHR) at 06:55 AM and arriving at Terminal 2 at Munich Airport (MUC) at 09:45 AM. The travel time is approximately 3 hours.'}

In [92]:
%%time

agent_executor.invoke("Which is the cheapest flight available from London to Munich on 18th September 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from London to Munich on 18th September 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-18T08:00:00",
    "departureDateTimeLatest": "2025-09-18T20:00:00"
  }
}
```

Thought: The human is asking for a flight from London to Munich on 18th September 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "LON",
    "destinationLocationCode": "MUC",
    "departureDateTimeEarliest": "2025-09-18T08:00:00",
    "departureDateTimeLatest": "2025-09-18T20:00:00"
  }
}
```


Observation: [{'price': {'total': '134.35', 'currency': 'EURO'}, 'segments': [{'departure': {

{'input': 'Which is the cheapest flight available from London to Munich on 18th September 2025?',
 'output': 'The cheapest flight available from London to Munich on 18th September 2025 is LUFTHANSA Flight Number 2483, departing from Terminal 2 at London Heathrow Airport (LHR) at 06:55 AM and arriving at Terminal 2 at Munich Airport (MUC) at 09:45 AM. The total price for this flight is 134.35 EURO.'}

In [93]:
%%time
agent_executor.invoke("Where is Oktoberfest 2025 going to be held? Provide venue details.")



> Entering new AgentExecutor chain...
Thought: The human is asking about Oktoberfest 2025 location and venue details.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 location"
}
```
Thought: The human is asking about Oktoberfest 2025 location and venue details.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest 2025 location"
}
```

Observation: Welcome to the Oktoberfest The festival of the City of Munich. In 2025, the 190th Oktoberfest will take place from September 20 to October 5 on the Theresienwiese.
Thought: 
Action:
```
{
  "action": "Final Answer",
  "action_input": "Oktoberfest 2025 will be held in Munich, Germany from September 20 to October 5 on the Theresienwiese."
}
```[/INST]

Final Answer: The final answer is Oktoberfest 2025 will be held in Munich, Germany from September 20 to October 5 on the Theresienwiese. I hope it is correct. Please let me know if I need to make any changes. Thank you for your patie

{'input': 'Where is Oktoberfest 2025 going to be held? Provide venue details.',
 'output': 'Oktoberfest 2025 will be held in Munich, Germany from September 20 to October 5 on the Theresienwiese.'}

In [98]:
%%time
agent_executor.invoke("What are the main events at Oktoberfest?")



> Entering new AgentExecutor chain...
Thought: The human is asking about Oktoberfest events.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest main events"
}
```[INST]

Thought: The human is asking about Oktoberfest events.
Action:
```
{
  "action": "Google Search tool",
  "action_input": "Oktoberfest main events"
}
```[INST]


Observation: Oktoberfest is the world's largest Volksfest, featuring a beer festival and a travelling carnival, and is held annually in Munich, Bavaria, from mid- or ... The festival of the City of Munich. In 2025, the 190th Oktoberfest will take place from September 20 to October 5 on the Theresienwiese. The festivities began October 12, 1810, and lasted nearly a week until October 17, ending with an exciting horse race. After such a spectacular party, the happy ... Here you can find out everything about the events and dates at Oktoberfest 2025 from 20 September to 5 October. The Oktoberfest is a two-week festival held each year i

{'input': 'What are the main events at Oktoberfest?',
 'output': 'Oktoberfest is a two-week festival held each year in Munich, Germany during late September and early October. It is attended by six million people each year. The festival features traditional German food, beer, and live music. Some of the popular activities at Oktoberfest include stein holding competitions, barrel rolling races, pretzel tossing games, and more.'}

In [99]:
%%time
agent_executor.invoke("Give me a 3-day itinerary for a trip to Munich during Oktoberfest.")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 3-day itinerary for Munich during Oktoberfest. I will use the Google Search tool to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Munich Oktoberfest itinerary"
}
```

Thought: The human is asking for a 3-day itinerary for Munich during Oktoberfest. I will use the Google Search tool to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Munich Oktoberfest itinerary"
}
```


Observation: Day 1-3: Munich, Germany. Oktoberfest, Marienplatz, Nymphenburg Palace, and the English Garden. · Day 4-5: Salzburg, Austria. Scenic train ride ... Our six-step-guide will help you plan the most perfect trip to the Oktoberfest in Munich. Learn which days to choose how to travel or what hotels to book. The BEST guide on attendi

{'input': 'Give me a 3-day itinerary for a trip to Munich during Oktoberfest.',
 'output': 'Day 1: Explore Marienplatz and the surrounding area. Visit the Viktualienmarkt food market and try some traditional German cuisine. In the evening, head to the Hofbräuhaus beer hall for a traditional Oktoberfest experience. Day 2: Visit the Nymphenburg Palace and its beautiful gardens. Take a stroll through the English Garden, one of the largest urban parks in the world. In the evening, enjoy some traditional German music and dance at a local tavern or beer garden. Day 3: Spend the day exploring the historic city center of Munich. Visit the iconic Neuschwanstein Castle, which is located just outside of Munich. Take a stroll through the charming streets of the old town, and visit some of the many shops, cafes, and restaurants that line the streets.'}

#### Scenario 4 - Paris Bastille Day

In [103]:
%%time
agent_executor.invoke({"input", "Can you give me the date of Bastille Day in 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for the date of Bastille Day in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day 2025"
}
```

Thought: The human is asking for the date of Bastille Day in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day 2025"
}
```


Observation: July 14th, 2025. Celebrate July 14th 2025 in Paris with an Unforgettable Cruise! Wondering how to make the most of your time in Paris on July 14th?
Thought: I have found the date of Bastille Day in 2025, which is July 14th. Now, I need to provide a final answer that includes all the relevant information about Bastille Day in Paris on July 14th, 2025.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The date of Bastille Day in 2025 is July 14th. Celebrate this day in Paris with an unforget

{'input': {'Can you give me the date of Bastille Day in 2025?', 'input'},
 'output': 'The date of Bastille Day in 2025 is July 14th. Celebrate this day in Paris with an unforgettable cruise! You can make the most of your time in Paris on July 14th by visiting popular tourist attractions, such as the Eiffel Tower and Notre-Dame Cathedral.'}

In [115]:
%%time
agent_executor.invoke({"input", "Which flights are available from New Delhi to Paris on 13th July 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for flight information from New Delhi to Paris on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-13T00:00:00",
    "departureDateTimeLatest": "2025-07-13T23:59:59"
  }
}
```

Thought: The human is asking for flight information from New Delhi to Paris on a specific date. I will use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-13T00:00:00",
    "departureDateTimeLatest": "2025-07-13T23:59:59"
  }
}
```


Observation: [{'price': {'total': '284.88', 'currency': 'EURO'}, 'segments': [{'departure': {'iataC

{'input': {'Which flights are available from New Delhi to Paris on 13th July 2025?',
  'input'},
 'output': ' I have found the available flights from New Delhi to Paris on 13th July 2025.\n\nAction:\n```\n{\n  "action": "Final Answer",\n  "action_input": "The available flights from New Delhi to Paris on 13th July 2025 are:\\n\\n1. Virgin Atlantic flight number 301, departing from New Delhi at 10:50 AM and arriving in London Heathrow at 16:05 PM.\\n\\n2. Virgin Atlantic flight number 6655, departing from London Heathrow at 19:45 PM and arriving in Paris Charles de Gaulle at 22:00 PM.\\n\\n3. Emirates flight number 511, departing from New Delhi at 11:00 AM and arriving in Dubai at 13:00 PM.\\n\\n4. Emirates flight number 75, departing from London Heathrow at 14:45 PM and arriving in Paris Charles de Gaulle at 17:05 PM.\\n\\n5. Emirates flight number 515, departing from Dubai at 23:55 PM on July 13th and arriving in Paris Charles de Gaulle at 09:25 AM on July 14th.\\n\\n6. Emirates flight

In [107]:
%%time
agent_executor.invoke("Which is the cheapest flight available from New Delhi to Paris on 12th July 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New Delhi to Paris on 12th July 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-12T00:00:00",
    "departureDateTimeLatest": "2025-07-12T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New Delhi to Paris on 12th July 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "DEL",
    "destinationLocationCode": "CDG",
    "departureDateTimeEarliest": "2025-07-12T00:00:00",
    "departureDateTimeLatest": "2025-07-12T23:59:59"
  }
}
```


Observation: [{'price': {'total': '284.88', 'currency': 'EURO'}, 'segments': [{'departure': {'iataC

{'input': 'Which is the cheapest flight available from New Delhi to Paris on 12th July 2025?',
 'output': 'The cheapest available flight from New Delhi to Paris on 12th July 2025 is DEL-CMB-CDG with a total price of 338.23 EURO.'}

In [108]:
%%time
agent_executor.invoke("Where are the Bastille Day celebrations in Paris going to be held? Provide location details.")



> Entering new AgentExecutor chain...
Thought: The human is asking about the location of Bastille Day celebrations in Paris. I need to use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day celebrations in Paris"
}
```

Thought: The human is asking about the location of Bastille Day celebrations in Paris. I need to use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Bastille Day celebrations in Paris"
}
```


Observation: Celebrations for Bastille Day in Paris on July 14, 2024 include a huge military ceremony and parade, Eiffel Tower concert and fireworks, Firemen's Balls, ... The oldest and largest military parade in Europe takes place on the morning of the 14th July on the Champs-Elysées in Paris. It's a festive and unifying event, with two days of exceptional free entertainment: military parade, cultural programme, Olympic torch relay,

{'input': 'Where are the Bastille Day celebrations in Paris going to be held? Provide location details.',
 'output': 'The Bastille Day military parade is the French military parade that has been held in the morning, every year in Paris, since 1880. The most festive celebrations; however, happen in Paris, where you can expect military parades and flyovers, fireworks, street parties, and all manner of pomp ... On 14 July 2024, in the run-up to the Paris Olympic and Paralympic Games, the famous and magnificent military parade will be taking place on the Avenue Foch.'}

In [109]:
%%time
agent_executor.invoke("What are the best places in Paris to view the Bastille Day fireworks?")



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on the best places to view the Bastille Day fireworks in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to view Bastille Day fireworks in Paris"
}
```

Thought: The human is asking for recommendations on the best places to view the Bastille Day fireworks in Paris. I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to view Bastille Day fireworks in Paris"
}
```


Observation: Monuments from which to admire the Bastille Day fireworks The Sacré-Cœur, the Montparnasse Tower or even the esplanade at Trocadéro are all places from which the Eiffel Tower is visible and where the atmosphere is certainly festive. You have to get their early to get a good spot.
Thought: I have found the best places to view the Basti

{'input': 'What are the best places in Paris to view the Bastille Day fireworks?',
 'output': 'The best places to view the Bastille Day fireworks in Paris are the Sacré-Cœur, the Montparnasse Tower or even the esplanade at Trocadéro. These locations offer a great view of the Eiffel Tower and are perfect for enjoying the festive atmosphere during Bastille Day.'}

In [110]:
%%time
agent_executor.invoke("Give me a 5-day itinerary for a trip to Paris that includes the Bastille Day celebrations.")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 5-day itinerary for Paris, including Bastille Day celebrations. I need to use the Google Search tool and knowledge base to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Paris 5-day itinerary Bastille Day"
}
```

Thought: The human is asking for a 5-day itinerary for Paris, including Bastille Day celebrations. I need to use the Google Search tool and knowledge base to find information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Paris 5-day itinerary Bastille Day"
}
```


Observation: To help you plan your trip to Paris and figure out what to see, what to do, where to stay, and where to eat, here's my suggested itinerary for a five-day visit. We plan to stay 5 nights in paris with our kids ( 8 and 12 years old). We nee

{'input': 'Give me a 5-day itinerary for a trip to Paris that includes the Bastille Day celebrations.',
 'output': 'Here is your 5-day Paris itinerary: Day 1: From Montmartre to the Opera · Day 2: Parisian Icons · Day 3: Latin Quarter & The East · Day 4: Day Trip to Versailles · Day 5: The West. You can stay in a hotel near the Eiffel Tower, such as the Pullman Paris Tour Eiffel or the Hotel Plaza Athenee. For dinner, you can try Le Grand Vefour, which is located in the heart of Paris and serves classic French cuisine. Alternatively, you can try Le Comptoir du Relais, which is a cozy bistro that serves traditional French dishes at reasonable prices.'}

In [111]:
%%time
agent_executor.invoke("Which hotels are recommended for staying near the Bastille Day celebration venues in Paris?")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations near the Bastille Day celebration venues in Paris. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "bastille day hotels paris"
}
```

Thought: The human is asking for hotel recommendations near the Bastille Day celebration venues in Paris. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "bastille day hotels paris"
}
```


Observation: Hotels near Bastille Day · 1. West End Hotel · 2. Hotel Marignan Champs-Elysées · 3. Hôtel Grand Powers · 4. Chambiges Elysees Hotel · 5. Hotel ... Hotels near Bastille Day · 31. Monsieur George Hotel & Spa · 32. Hotel San Regis · 33. Hôtel Lord Byron · 34. O.lysée Hôtel · 35. The Peninsula Paris · 36. Le ... Where to stay in Paris · Villa Beaumarchais · Pullman Paris Tour Eiffel · ibis Paris 

{'input': 'Which hotels are recommended for staying near the Bastille Day celebration venues in Paris?',
 'output': "The recommended hotels for staying near the Bastille Day celebration venues in Paris are: Hôtel de Banville or La Comtesse. However, if you're looking for more options, consider staying at Villa Beaumarchais, Pullman Paris Tour Eiffel, ibis Paris Tour Eiffel Cambronne 15ème, citizenM Paris Opera, Hotel Bowmann, InterContinental Paris le Grand, an IHG Hotel, Fraser Suites le Claridge Champs-Elysées. These hotels offer a range of amenities and services to suit different budgets and preferences."}

#### Scenario 5 - India Holi

In [119]:
%%time
agent_executor.invoke({"input", "I want to experience the Holi festival in India. What is the significance of this festival?"})



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "significance of Holi festival in India"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "significance of Holi festival in India"
}
```


Observation: The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive. People pay or forgive debts, as well as deal anew with those in their lives.
Thought: the human asked about the significance of Holi festival in India, and I used Google Search tool to find the answer.

Action:```{
  "action": "Final Answer",
  "action_input": "The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive

{'input': {'I want to experience the Holi festival in India. What is the significance of this festival?',
  'input'},
 'output': 'The Holi festival has a cultural significance among various Hindu traditions of the Indian subcontinent. It is the festive day to end and rid oneself of past errors, to end conflicts by meeting others, a day to forget and forgive. People pay or forgive debts, as well as deal anew with those in their lives.'}

In [120]:
%%time
agent_executor.invoke({"input", "When is Holi celebrated in 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about the date of Holi in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi celebration date in 2025"
}
```

Thought: The human is asking about the date of Holi in 2025. I can use the Google Search tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi celebration date in 2025"
}
```


Observation: However, with the course of time, the festival has spread its wings to other parts of the world, including England, USA, and Australia, among others. In 2025, Holi will be celebrated on 14th March, Friday.
Thought: the human asked about Holi celebration date in 2025, and I used the Google Search tool to find this information.

Action:
```
{
  "action": "Final Answer",
  "action_input": "Holi will be celebrated on 14th March, Friday in 2025."
}
```[/INST] 

Final Answer: The final answer is

{'input': {'When is Holi celebrated in 2025?', 'input'},
 'output': 'Holi will be celebrated on 14th March, Friday in 2025.'}

In [121]:
%%time
agent_executor.invoke({"input", "Can you give me the best places to celebrate Holi in India?"})



> Entering new AgentExecutor chain...
Thought: The human is asking for recommendations on places to celebrate Holi in India. I can use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in India"
}
```

Thought: The human is asking for recommendations on places to celebrate Holi in India. I can use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in India"
}
```


Observation: Here's a list of the top destinations in India to travel to, celebrate the festival of colours, and make the most of this long weekend.  1. Mathura and Vrindavan, Uttar Pradesh. ...  2. Shantiniketan, West Bengal. ...  3. Udaipur and Jaipur, Rajasthan. ...  4. Varanasi, Uttar Pradesh. ...  5. Anandpur Sahib, Punjab. ...  6. Sangla, Himachal Pradesh. ...  7. Pushkar, Rajasthan.
Thought: I used the Google Search to

{'input': {'Can you give me the best places to celebrate Holi in India?',
  'input'},
 'output': 'The top destinations in India to travel to, celebrate the festival of colours, and make the most of this long weekend are: Mathura and Vrindavan, Uttar Pradesh; Shantiniketan, West Bengal; Udaipur and Jaipur, Rajasthan; Varanasi, Uttar Pradesh; Anandpur Sahib, Punjab; Sangla, Himachal Pradesh; Pushkar, Rajasthan.'}

In [122]:
%%time
agent_executor.invoke({"input", "What are the main events during the Holi festival?"})



> Entering new AgentExecutor chain...
Action:```{
  "action": "Google Search tool",
  "action_input": "Holi festival main events"
}
```

Action:```{
  "action": "Google Search tool",
  "action_input": "Holi festival main events"
}
```


Observation: The main day of Holi is marked by cheerful celebrations. People throw colored powders and water at each other, dance to music, and enjoy festive foods. Streets are filled with laughter, music, and vibrant colors.
Thought: the human asked about the main events during the Holi festival, and I used the Google Search tool to find the answer.

Action:```{
  "action": "Final Answer",
  "action_input": "The main day of Holi is marked by cheerful celebrations. People throw colored powders and water at each other, dance to music, and enjoy festive foods. Streets are filled with laughter, music, and vibrant colors."
}
```

 the human asked about the main events during the Holi festival, and I used the Google Search tool to find the answer.

Action:

{'input': {'What are the main events during the Holi festival?', 'input'},
 'output': 'The main day of Holi is marked by cheerful celebrations. People throw colored powders and water at each other, dance to music, and enjoy festive foods. Streets are filled with laughter, music, and vibrant colors.'}

In [123]:
%%time
agent_executor.invoke({"input", "How can I participate in Holi celebrations in India?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about participating in Holi celebrations in India. I need to provide information on how to participate in the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "How to participate in Holi celebrations in India"
}
```

Thought: The human is asking about participating in Holi celebrations in India. I need to provide information on how to participate in the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "How to participate in Holi celebrations in India"
}
```


Observation: Bonfires are lit and people gather around while eating traditional foods. The second day of the festival is when people celebrate by gathering in groups and splashing brilliantly colored dyes at each other. Every year, Holi is observed on the last full moon day of the Hindu lunisolar calendar month.
Thought: the human asked about participating in Holi celebrations in India, and I provided inform

{'input': {'How can I participate in Holi celebrations in India?', 'input'},
 'output': 'To participate in Holi celebrations in India, you can join a group of people who are celebrating the festival. You can also purchase colored powders or dyes from local vendors and use them to play with during the celebration. Additionally, you can attend a Holi celebration event at a local temple or community center.'}

In [124]:
%%time
agent_executor.invoke("Can you find the cheapest flight from San Francisco to Delhi on a single day on 8th March 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a single flight from San Francisco to Delhi on 8th March 2025. I need to use the single_flight_search tool.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-03-08T00:00:00",
    "departureDateTimeLatest": "2025-03-08T23:59:59"
  }
}
```

Thought: The human is asking for a single flight from San Francisco to Delhi on 8th March 2025. I need to use the single_flight_search tool.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "DEL",
    "departureDateTimeEarliest": "2025-03-08T00:00:00",
    "departureDateTimeLatest": "2025-03-08T23:59:59"
  }
}
```


Observation: [{'price': {'total': '460.90', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'SFO', 'terminal': '1', 'at': '2025-03-08T09:15:

{'input': 'Can you find the cheapest flight from San Francisco to Delhi on a single day on 8th March 2025?',
 'output': 'The cheapest flight from San Francisco to Delhi on a single day on 8th March 2025 is $460.90 with Air India.'}

In [130]:
%%time
agent_executor.invoke("Which is the cheapest flight available from San Francisco to Varanasi on 6th March 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from San Francisco to Varanasi on 6th March 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "VNS",
    "departureDateTimeEarliest": "2025-03-06T00:00:00",
    "departureDateTimeLatest": "2025-03-06T23:59:59"
  }
}
```

Thought: The human is asking for a flight from San Francisco to Varanasi on 6th March 2025. I need to use the single_flight_search tool to find the cheapest available flight.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "SFO",
    "destinationLocationCode": "VNS",
    "departureDateTimeEarliest": "2025-03-06T00:00:00",
    "departureDateTimeLatest": "2025-03-06T23:59:59"
  }
}
```


Observation: [{'price': {'total': '499.77', 'currency': 'EURO'}, 'segments': [{'depar

{'input': 'Which is the cheapest flight available from San Francisco to Varanasi on 6th March 2025?',
 'output': 'The cheapest available flight from San Francisco to Varanasi on 6th March 2025 is Air India Flight Number 406, departing from San Francisco at 20:00 on 6th March 2025 and arriving in Varanasi at 12:00 on 8th March 2025. The total price for this flight is 519.64 EURO.'}

In [126]:
%%time
agent_executor.invoke("Where are the best places to celebrate Holi in Delhi? Provide venue details.")



> Entering new AgentExecutor chain...
Thought: The human is asking for Holi celebration venues in Delhi. I need to find the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in Delhi"
}
```

Thought: The human is asking for Holi celebration venues in Delhi. I need to find the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Best places to celebrate Holi in Delhi"
}
```


Observation: The Delhi Holi festival locations around India Gate feature live music, dance performances, and elaborate food stalls, making it one of the best places to celebrate Holi in Delhi. The majestic Red Fort is another popular venue for Holi celebrations in Delhi.
Thought: finding the best places to celebrate Holi in Delhi.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The best places to celebrate Holi in Delhi are around India Gate and at the Red Fo

{'input': 'Where are the best places to celebrate Holi in Delhi? Provide venue details.',
 'output': 'The best places to celebrate Holi in Delhi are around India Gate and at the Red Fort. These locations offer live music, dance performances, elaborate food stalls, and a majestic atmosphere making them ideal for celebrating Holi in Delhi.'}

In [127]:
%%time
agent_executor.invoke("What traditional foods should I try during the Holi festival?")



> Entering new AgentExecutor chain...
Thought: The Holi festival is a traditional Hindu celebration that marks the arrival of spring. It's known for its vibrant colors, music, and dance.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Traditional foods to try during Holi festival"
}
```

Thought: The Holi festival is a traditional Hindu celebration that marks the arrival of spring. It's known for its vibrant colors, music, and dance.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Traditional foods to try during Holi festival"
}
```


Observation: 11 foods & drinks of holi celebration ___Gujiya. Crispy on the outside with a delicious stuffing on the inside, this dessert is likened to cornish pasties, only sweet! ... ___Thandai. ... ___Lassi. ... ___Rasmalai. ... ___Malpua. ... ___Barfi. ... ___Badam phirni. ... ___Dahi bhalla/Dahi vada.
Thought: the human asked about traditional foods to try during the Holi festival, and I provided a list of 11

{'input': 'What traditional foods should I try during the Holi festival?',
 'output': 'The traditional foods you should try during the Holi festival are: Gujiya, Thandai, Lassi, Rasmalai, Malpua, Barfi, Badam phirni, and Dahi bhalla/Dahi vada.'}

In [132]:
%%time
agent_executor.invoke("Give me a 2-day itinerary for a trip to India during the Holi festival.")



> Entering new AgentExecutor chain...
Thought: The Holi festival is a significant event in India, and it's best to plan the itinerary around this festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi festival itinerary India"
}
```

Thought: The Holi festival is a significant event in India, and it's best to plan the itinerary around this festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Holi festival itinerary India"
}
```


Observation: Discover the vibrant Holi festival in India with our complete guide, featuring top celebration spots and essential travel and safety tips. Day 1: Arrival in Delhi We will arrange for you to be picked up from the airport if requested. There is a pre-tour meet 'n' greet with Nathan at the hotel ... 1 guided yoga class; A safari through Ranthambore National Park; Holika Dahan fire ritual to welcome Holi; Poolside Holi celebration at a boutique palace ... Currently planning a route to include some

{'input': 'Give me a 2-day itinerary for a trip to India during the Holi festival.',
 'output': "The detailed itinerary for the trip to India during the Holi festival is as follows: Day 1: Arrival in Delhi We will arrange for you to be picked up from the airport if requested. There is a pre-tour meet 'n' greet with Nathan at the hotel ... 1 guided yoga class; A safari through Ranthambore National Park; Holika Dahan fire ritual to welcome Holi; Poolside Holi celebration at a boutique palace ... Day 2: Delhi Local - Agra · Sunrise visit to the Taj Mahal · Other activities such as yoga classes, safari through Ranthambore National Park, Holika Dahan fire ritual to welcome Holi, and poolside Holi celebration at a boutique palace."}

In [129]:
%%time
agent_executor.invoke("Which hotels are recommended for staying in Delhi during Holi celebrations?")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations in Delhi during Holi celebrations. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Delhi hotels recommended during Holi celebrations"
}
```

Thought: The human is asking for hotel recommendations in Delhi during Holi celebrations. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "Delhi hotels recommended during Holi celebrations"
}
```


Observation: ___Delhi Bed and Breakfast. 388 reviews. New Delhi, India. ... ___Stay Inn. 3 reviews. New Delhi, India. ... ___The Lalit New Delhi. 12,045 reviews. New Delhi, India. ... ___Hotel Amax Inn. 1,169 reviews. New Delhi, India. ... ___Home at F37. 325 reviews. ... ___The Oberoi, New Delhi. 4,019 reviews. ... ___Hotel Hari Piorko. 2,427 reviews. ... ___Shangri-La Eros New Delhi. 5,18

{'input': 'Which hotels are recommended for staying in Delhi during Holi celebrations?',
 'output': 'The recommended hotels for staying in Delhi during Holi celebrations are: Delhi Bed and Breakfast, Stay Inn, The Lalit New Delhi, Hotel Amax Inn, Home at F37, The Oberoi, New Delhi, Hotel Hari Piorko, Shangri-La Eros New Delhi. These hotels have been rated based on their reviews and ratings.'}

#### Scenario 6 - Spain Tomatina festival

In [133]:
%%time
agent_executor.invoke({"input", "I want to attend La Tomatina festival in Spain. What is the significance of this event?"}) 



> Entering new AgentExecutor chain...
Thought: The human wants to attend La Tomatina festival in Spain. I need to provide information about the significance of this event.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival significance"
}
```

Thought: The human wants to attend La Tomatina festival in Spain. I need to provide information about the significance of this event.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival significance"
}
```


Observation: La Tomatina is no ordinary festival. Its origins trace back to a simple food fight among friends in the mid-20th century, with tomatoes being the weapon of choice. Over the years, this playful encounter evolved into an official event, where hundreds of tons of ripe tomatoes are now flung with unrestrained joy.
Thought: the human wants to attend La Tomatina festival in Spain, and I provided information about the significance of this event.

Action:
```
{


{'input': {'I want to attend La Tomatina festival in Spain. What is the significance of this event?',
  'input'},
 'output': 'La Tomatina is a unique festival that originated from a simple food fight among friends. It has evolved into an official event where hundreds of tons of ripe tomatoes are flung with unrestrained joy.'}

In [134]:
%%time
agent_executor.invoke({"input", "When is La Tomatina celebrated in 2025?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about the date of La Tomatina in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025"
}
```

Thought: The human is asking about the date of La Tomatina in 2025. I need to use a tool to find this information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025"
}
```


Observation: La Tomatina is a food fight festival held on the last Wednesday of August each year in the town of Buñol (history) near to Valencia in Spain ... This Tomatina is celebrated on August 24th and it is the opportunity for kids from ages 4 to 12 to enjoy the particular Tomatina party. READ MORE ... ... La Tomatina 2025: Wednesday, 27th of August La Tomatina 2026: Wednesday, 26th of August La Tomatina 2027: Wednesday, 25th of August. Other La Tomatina ... 2025 date, August 27 (2025-08-27). 2026 date, August 26 (2026-08-26). Freque

{'input': {'When is La Tomatina celebrated in 2025?', 'input'},
 'output': 'La Tomatina is celebrated on Wednesday, August 27th, 2025.'}

In [135]:
%%time
agent_executor.invoke({"input", "Can you give me the dates for La Tomatina festival in Buñol, Spain?"})



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is a popular event in Buñol, Spain. It takes place every year on the last Wednesday of August.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival dates"
}
```

Thought: The La Tomatina festival is a popular event in Buñol, Spain. It takes place every year on the last Wednesday of August.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival dates"
}
```


Observation: The festival is held annually on the last Wednesday of August, in the case of 2024 on 28th August. It starts around 11am and lasts for about an hour, featuring a massive tomato fight in the town square.
Thought: the La Tomatina festival is held annually on the last Wednesday of August, in Buñol, Spain. The festival starts around 11am and lasts for about an hour.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The La Tomatina festival is held annually on the last W

{'input': {'Can you give me the dates for La Tomatina festival in Buñol, Spain?',
  'input'},
 'output': 'The La Tomatina festival is held annually on the last Wednesday of August, in Buñol, Spain. The festival starts around 11am and lasts for about an hour.'}

In [144]:
%%time
agent_executor.invoke({"input", "How can I book tickets for La Tomatina festival in Spain?"})



> Entering new AgentExecutor chain...
Thought: The human is asking about booking tickets for La Tomatina festival in Spain. I need to use the Google Search tool to find information on how to book tickets for the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "How to book tickets for La Tomatina festival in Spain"
}
```

Thought: The human is asking about booking tickets for La Tomatina festival in Spain. I need to use the Google Search tool to find information on how to book tickets for the festival.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "How to book tickets for La Tomatina festival in Spain"
}
```


Observation: Buy your Tomatina ticket in 3 easy steps: choose your city, pay in a secure way and finally print your ticket to exchange it for the official wristband to ... Buy official Ticket Tomatina festival 2024. Come to Buñol to launch tomatoes, single ticket or with transfer from Valencia, Madrid and Barcelona ... Buy Ticket

{'input': {'How can I book tickets for La Tomatina festival in Spain?',
  'input'},
 'output': 'The human can buy official tickets for La Tomatina festival in Spain by visiting the official website and following the instructions to purchase tickets online. The ticket prices vary depending on the package chosen, but they typically range from $50 to $200 per person.'}

In [137]:
%%time
agent_executor.invoke("Which flight is available from New York to Valencia so that I can arrive on August 24, 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the available flights.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```


Observation: [{'price': {'total': '360.90', 'currency': 'EURO'}, 'segments': [{'departure': {'iataCode': 'JFK

{'input': 'Which flight is available from New York to Valencia so that I can arrive on August 24, 2025?',
 'output': 'The final answer is that there are multiple flights available from New York to Valencia on August 24, 2025. The first flight is operated by Azores Airlines and has the following details: Flight Number: 246, Departure Time: 20:55:00, Arrival Time: 06:50:00, Terminal: 1. The second flight is also operated by Azores Airlines and has the following details: Flight Number: 120, Departure Time: 11:40:00, Arrival Time: 17:35:00, Terminal: 1.'}

In [139]:
%%time
agent_executor.invoke("Which is the cheapest flight available from New York to Valencia on August 24, 2025?")



> Entering new AgentExecutor chain...
Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```

Thought: The human is asking for a flight from New York to Valencia on August 24, 2025. I need to use the single_flight_search tool to find the cheapest flight available.

Action:
```
{
  "action": "single_flight_search",
  "action_input": {
    "originLocationCode": "JFK",
    "destinationLocationCode": "VLC",
    "departureDateTimeEarliest": "2025-08-24T00:00:00",
    "departureDateTimeLatest": "2025-08-24T23:59:59"
  }
}
```


Observation: [{'price': {'total': '360.90', 'currency': 'EURO'}, 'segments': [{'departure': {

{'input': 'Which is the cheapest flight available from New York to Valencia on August 24, 2025?',
 'output': 'The cheapest flight available from New York to Valencia on August 24, 2025 is $360.90 with a departure time of 20:55:00 and an arrival time of 06:50:00.'}

In [140]:
%%time
agent_executor.invoke("Where is La Tomatina 2025 going to be held? Provide venue details.")



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. I need to find out where it will be held in 2025.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025 venue"
}
```

Thought: The La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. I need to find out where it will be held in 2025.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina 2025 venue"
}
```


Observation: La Tomatina 2025 - Information, Accommodation & Tickets La Tomatina is a food fight festival held on the last Wednesday of August each year in the town of Buñol (history) near to Valencia in Spain.
Thought: the La Tomatina festival is held annually in Buñol, a small town in the Valencia region of Spain. The exact location for 2025 has not been specified yet.

Action:
```
{
  "action": "Final Answer",
  "action_input": "The La Toma

{'input': 'Where is La Tomatina 2025 going to be held? Provide venue details.',
 'output': 'The La Tomatina festival will be held in Buñol, a small town in the Valencia region of Spain. However, the exact location for 2025 has not been specified yet.'}

In [141]:
%%time
agent_executor.invoke("What are the main events during La Tomatina festival?")



> Entering new AgentExecutor chain...
Thought: The La Tomatina festival is a popular event in Spain, known for its unique and entertaining activities. To answer the human's question about the main events during La Tomatina festival, I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival main events"
}
```

Thought: The La Tomatina festival is a popular event in Spain, known for its unique and entertaining activities. To answer the human's question about the main events during La Tomatina festival, I will use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival main events"
}
```


Observation: The main event of La Tomatina is a massive tomato fight that lasts for about an hour. At exactly 11 a.m., a signal sounds, and trucks loaded with overripe tomatoes roll into the town square. Participants eagerl

{'input': 'What are the main events during La Tomatina festival?',
 'output': 'The main event of La Tomatina is a massive tomato fight that lasts for about an hour. At exactly 11 a.m., a signal sounds, and trucks loaded with overripe tomatoes roll into the town square. Participants eagerly scoop up the tomatoes, and the battle begins.'}

In [142]:
%%time
agent_executor.invoke("Give me a 5-day itinerary for a trip to Buñol, Spain during La Tomatina.")



> Entering new AgentExecutor chain...
Thought: The human is asking for a 5-day itinerary for La Tomatina in Buñol, Spain. I need to provide the human with detailed information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina itinerary Buñol Spain"
}
```

Thought: The human is asking for a 5-day itinerary for La Tomatina in Buñol, Spain. I need to provide the human with detailed information on tourist attractions, must-visit places, and hotels with ratings.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina itinerary Buñol Spain"
}
```


Observation: A review of the crazy festival known as La Tomatina that takes place in Spain in August, including tips, tricks, & info on how to attend. This is all you need to know to plan your La Tomatina Festival in Spain. From stay, transportation, what to wear and all you need. La Tomatina is a food fight festival held on 

{'input': 'Give me a 5-day itinerary for a trip to Buñol, Spain during La Tomatina.',
 'output': 'Here is your detailed 5-day itinerary for La Tomatina in Buñol, Spain:\n\nDay 1: Arrival in Valencia and transfer to Buñol\n\nDay 2: Explore the town of Buñol and its surroundings\n\nDay 3: La Tomatina festival day!\n\nDay 4: Relaxation day in Buñol or explore nearby Valencia\n\nDay 5: Departure from Buñol and transfer back to Valencia'}

In [143]:
%%time
agent_executor.invoke("Which hotels are recommended for staying near Buñol during La Tomatina festival?")



> Entering new AgentExecutor chain...
Thought: The human is asking for hotel recommendations near Buñol during La Tomatina festival. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival hotels in Buñol"
}
```

Thought: The human is asking for hotel recommendations near Buñol during La Tomatina festival. I need to use the Google Search tool to find relevant information.

Action:
```
{
  "action": "Google Search tool",
  "action_input": "La Tomatina festival hotels in Buñol"
}
```


Observation: Hotels near La Tomatina ; 1. Holiday Inn Express Valencia - Bonaire, an IHG hotel · 357 reviews. 16.4 miles ; 2. Travelodge Valencia ... How to get yourself from Valencia or Barcelona to Bunol and packages with official entry tickets and accommodation for the famous La Tomatina Tomato fight. Resorts in Buñol which are exceptionally rated by group vacationers are LA TRAVIATA - Bunol, Casa Cerva

{'input': 'Which hotels are recommended for staying near Buñol during La Tomatina festival?',
 'output': 'The recommended hotels for staying near Buñol during La Tomatina festival are: Holiday Inn Express Valencia - Bonaire, an IHG hotel; Travelodge Valencia; Ibis Budget Valencia Alcasser. These hotels offer a range of amenities and services to suit different budgets and preferences.'}

### Streamlit UI Demo
Navigate to the directory where the Streamlit file is located, then run the following commands in the terminal within the activated environment.

! streamlit run Final_AI_Travel_Agent_streamlit.py